# pa2-dev — setup checks + GPU smoke tests (INTERACTIVE, ~15 min)
Settings: GPU T4 x2, Internet on. Run cells one by one. Nothing here is a submitted result.
Purpose: catch CUDA / memory / path problems in minutes instead of after a 2-hour commit.

In [ ]:
# ---- Setup: identical first cell in every notebook (a commit starts on a fresh machine) ----
GH_USER, REPO = "mohdddd", "small-model-alignment-starter"
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/{GH_USER}/{REPO}.git repo
%cd /kaggle/working/repo
!git log --oneline -1
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps
import os, sys, json, glob, shutil, subprocess
sys.path.insert(0, "/kaggle/working/repo/src")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("outputs", exist_ok=True)

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch, transformers, datasets
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/ 2>&1 | tail -3

In [ ]:
!python scripts/prepare_data.py --which sft hh alpaca --out_dir outputs/smoke_data --sft_train 64 --pref_train 64 --n_val 16

SFT smoke: 3 optimizer steps at the real batch shape (2 x 512). Check `peak_gpu_mem_gb` < ~14.

In [ ]:
!python scripts/train_sft.py --train_file outputs/smoke_data/sft_train.jsonl --val_file outputs/smoke_data/sft_val.jsonl --out_dir outputs/smoke_sft --max_steps 3 --grad_accum 4 --eval_every 3 --log_every 1 --no_save
print(open('outputs/smoke_sft/summary.json').read())

DPO smoke (base model stands in for the SFT checkpoint): policy cuda:0, reference cuda:1. Step-0 eval must show val_loss = 0.6931, reward_acc = 0.0, margin = 0.0.

In [ ]:
!python scripts/train_dpo.py --sft_path Qwen/Qwen2.5-0.5B --train_file outputs/smoke_data/hh_train.jsonl --val_file outputs/smoke_data/hh_val.jsonl --out_dir outputs/smoke_dpo --max_steps 3 --grad_accum 4 --eval_every 3 --log_every 1 --no_save
print(open('outputs/smoke_dpo/summary.json').read())

Generation sanity check on 3 prompts (base model, greedy + repetition_penalty 1.1).

In [ ]:
!python scripts/generate.py --model Qwen/Qwen2.5-0.5B --eval_set outputs/smoke_data/alpaca_eval_300_seed42.json --out outputs/smoke_gen/base.json --generator qwen2.5-0.5b-base --limit 3 --max_new_tokens 200

If all cells passed: **Quick Save**, stop this session (to free GPU quota), then commit `01_sft`.